In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


## Data Collection and Processing

In [ ]:
# ── FIX: Update this path to wherever your train CSV is located ──────────────
# If running on Kaggle, use: '/kaggle/input/titanic/train.csv'
# If running locally, set the correct path below.
titanic_data = pd.read_csv(r"C:\Users\DELL\OneDrive\Desktop\MLPROJECTS\1. TITANIC\traintitanic.csv")


In [ ]:
titanic_data.head()

In [ ]:
titanic_data.shape

In [ ]:
titanic_data.info()

In [ ]:
titanic_data.isnull().sum()

## Handling Missing Values

> **Note:** We do **not** drop `Cabin` here yet — we need it in the Feature Engineering section to extract `Deck`. We drop it at the end after extraction.

In [ ]:
# Fill missing Age with median
titanic_data["Age"] = titanic_data["Age"].fillna(titanic_data["Age"].median())


In [ ]:
# Fill missing Embarked with mode
titanic_data['Embarked'].fillna(titanic_data['Embarked'].mode()[0], inplace=True)


In [ ]:
titanic_data.isnull().sum()

## Data Analysis

In [ ]:
titanic_data.describe()

In [ ]:
titanic_data['Survived'].value_counts()

## Data Visualisation

In [ ]:
sns.set()

In [ ]:
sns.countplot(x='Survived', data=titanic_data)

In [ ]:
sns.countplot(x='Sex', data=titanic_data)

In [ ]:
sns.countplot(x='Sex', hue='Survived', data=titanic_data)

In [ ]:
sns.countplot(x='Pclass', data=titanic_data)

In [ ]:
sns.countplot(x='Pclass', hue='Survived', data=titanic_data)

## Encoding Categorical Columns

In [ ]:
titanic_data['Sex'].value_counts()

In [ ]:
titanic_data['Embarked'].value_counts()

In [ ]:
titanic_data.replace({'Sex': {'male': 0, 'female': 1},
                      'Embarked': {'S': 0, 'C': 1, 'Q': 2}}, inplace=True)
titanic_data.head()


## Feature Engineering

Here we create new, more informative columns from the existing data.  
Each new column gives the model extra signal that raw columns don't provide directly.

> **FIX:** `Cabin` is **not** dropped before this section — all deck extraction happens here cleanly, then we drop `Cabin` at the end.


In [ ]:
# ── 1. EXTRACT DECK FROM CABIN ─────────────────────────────────────────────
# Cabin column is still present here (we did NOT drop it earlier — that was the bug).
# We extract the first letter (deck letter: A, B, C ... G, T) before dropping Cabin.

deck_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'T': 8, 'Unknown': 0}

titanic_data['Deck'] = titanic_data['Cabin'].str[0]
titanic_data['Deck'] = titanic_data['Deck'].fillna('Unknown')
titanic_data['Deck'] = titanic_data['Deck'].map(deck_map).fillna(0).astype(int)

# Now drop Cabin — we've extracted everything we need from it
titanic_data.drop(columns='Cabin', inplace=True)

print('Deck value counts:')
print(titanic_data['Deck'].value_counts())


In [ ]:
# ── 2. FAMILY SIZE & IS ALONE ────────────────────────────────────────────────
# SibSp = siblings/spouses aboard; Parch = parents/children aboard
# +1 because we count the passenger themselves.

titanic_data['FamilySize'] = titanic_data['SibSp'] + titanic_data['Parch'] + 1
titanic_data['IsAlone'] = (titanic_data['FamilySize'] == 1).astype(int)

print('FamilySize distribution:')
print(titanic_data['FamilySize'].value_counts().sort_index())
print(f"\nTravelling alone: {titanic_data['IsAlone'].sum()} passengers")


In [ ]:
# ── 3. EXTRACT TITLE FROM NAME ───────────────────────────────────────────────
# Every name looks like: 'Braund, Mr. Owen Harris'
# regex extracts the word just before a dot.

titanic_data['Title'] = titanic_data['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
print('All titles found:')
print(titanic_data['Title'].value_counts())


In [ ]:
title_map = {
    'Mr': 1,
    'Miss': 2,
    'Mrs': 3,
    'Master': 4,
    'Dr': 5,
    'Rev': 6,
}
titanic_data['Title'] = titanic_data['Title'].map(title_map).fillna(0).astype(int)

print('Title after mapping:')
print(titanic_data['Title'].value_counts())


In [ ]:
# ── 4. FARE PER PERSON ───────────────────────────────────────────────────────
# Tickets were sometimes shared — divide Fare by FamilySize for individual wealth proxy.

titanic_data['FarePerPerson'] = titanic_data['Fare'] / titanic_data['FamilySize']
titanic_data['FarePerPerson'] = titanic_data['FarePerPerson'].fillna(0)

print('FarePerPerson - first 5 rows:')
print(titanic_data[['Fare', 'FamilySize', 'FarePerPerson']].head())


In [ ]:
# ── 5. AGE × PCLASS INTERACTION ──────────────────────────────────────────────
# Combines age and class into a single interaction feature.

titanic_data['AgePclass'] = titanic_data['Age'] * titanic_data['Pclass']

print('AgePclass - first 5 rows:')
print(titanic_data[['Age', 'Pclass', 'AgePclass']].head())


In [ ]:
# ── Summary: all new features ─────────────────────────────────────────────────
new_features = ['Deck', 'FamilySize', 'IsAlone', 'Title', 'FarePerPerson', 'AgePclass']
print('New feature columns added:')
titanic_data[new_features].head(10)


### Quick Visualisation — do our new features relate to survival?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.barplot(x='Title', y='Survived', data=titanic_data, ax=axes[0,0])
axes[0,0].set_title('Title vs Survival')
axes[0,0].set_xlabel('Title (1=Mr, 2=Miss, 3=Mrs, 4=Master)')

sns.barplot(x='FamilySize', y='Survived', data=titanic_data, ax=axes[0,1])
axes[0,1].set_title('Family Size vs Survival')

sns.barplot(x='IsAlone', y='Survived', data=titanic_data, ax=axes[0,2])
axes[0,2].set_title('Is Alone vs Survival (0=with family)')

sns.barplot(x='Deck', y='Survived', data=titanic_data, ax=axes[1,0])
axes[1,0].set_title('Deck vs Survival (0=Unknown)')

sns.boxplot(x='Survived', y='FarePerPerson', data=titanic_data, ax=axes[1,1])
axes[1,1].set_title('Fare Per Person vs Survival')
axes[1,1].set_ylim(0, 100)

sns.boxplot(x='Survived', y='AgePclass', data=titanic_data, ax=axes[1,2])
axes[1,2].set_title('Age × Pclass vs Survival')

plt.tight_layout()
plt.show()


## Separating Features and Target

In [ ]:
# Drop raw columns replaced by engineered features:
#   SibSp, Parch  → FamilySize, IsAlone
#   Fare          → FarePerPerson
#   Name, Ticket  → Title extracted from Name; Ticket not useful

X = titanic_data.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Survived', 'SibSp', 'Parch', 'Fare'],
    axis=1
)
Y = titanic_data['Survived']

print('Features used for model training:')
print(list(X.columns))
print(f'\nShape: {X.shape}')


In [ ]:
print(X)

## Splitting the Data into Training and Test Sets

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2, stratify=Y
)
print(X.shape, X_train.shape, X_test.shape)


## Pipelining

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])


## Model Training

In [ ]:
pipeline.fit(X_train, Y_train)

## Model Evaluation

In [ ]:
# Training accuracy
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print('Accuracy on training data:', training_data_accuracy * 100)


In [ ]:
# Test accuracy
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)
print('Accuracy on test data:', test_data_accuracy * 100)


In [ ]:
print(confusion_matrix(Y_test, X_test_prediction))
print('Precision:', precision_score(Y_test, X_test_prediction))
print('Recall:   ', recall_score(Y_test, X_test_prediction))


In [ ]:
# Cross Validation
cv_scores = cross_val_score(pipeline, X, Y, cv=5)
print('CV Scores:', cv_scores)
print('Average CV Score:', cv_scores.mean())


## Kaggle Submission — Predict on test.csv

> **FIX:** All feature engineering steps are applied to `test_data` in the same order as training.  
> `deck_map` and `title_map` are already defined above — no re-definition needed.  
> `Deck` mapping now includes `'T': 8` to cover all possible cabin letters in test data, preventing NaN values that would break the pipeline.


In [ ]:
# ── 1. Load Kaggle test data ─────────────────────────────────────────────────
# Update path if needed. On Kaggle: '/kaggle/input/titanic/test.csv'
test_data = pd.read_csv(r"C:\Users\DELL\OneDrive\Desktop\MLPROJECTS\1. TITANIC\test.csv")

print(f'Test data shape: {test_data.shape}')  # Should be (418, 11)


In [ ]:
# ── 2. Preprocess test data (must match training exactly) ────────────────────
test_data['Age'].fillna(test_data['Age'].median(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].median(), inplace=True)
test_data['Embarked'].fillna(test_data['Embarked'].mode()[0], inplace=True)

test_data.replace({'Sex': {'male': 0, 'female': 1},
                   'Embarked': {'S': 0, 'C': 1, 'Q': 2}}, inplace=True)


In [ ]:
# ── 3. Apply the same feature engineering steps ──────────────────────────────

# A. Deck from Cabin (uses deck_map defined in training FE section)
test_data['Deck'] = test_data['Cabin'].str[0]
test_data['Deck'] = test_data['Deck'].fillna('Unknown')
test_data['Deck'] = test_data['Deck'].map(deck_map).fillna(0).astype(int)

# B. FamilySize & IsAlone
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

# C. Title from Name (uses title_map defined in training FE section)
test_data['Title'] = test_data['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
test_data['Title'] = test_data['Title'].map(title_map).fillna(0).astype(int)

# D. FarePerPerson
test_data['FarePerPerson'] = test_data['Fare'] / test_data['FamilySize']
test_data['FarePerPerson'] = test_data['FarePerPerson'].fillna(0)

# E. AgePclass
test_data['AgePclass'] = test_data['Age'] * test_data['Pclass']


In [ ]:
# ── 4. Prepare feature matrix — drop same raw columns as in training ──────────
X_test_kaggle = test_data.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Fare'],
    axis=1
)

print('Test feature columns:', list(X_test_kaggle.columns))
print(f'Shape: {X_test_kaggle.shape}')  # Should be (418, 10)

# Sanity check — columns must exactly match training columns
assert list(X_test_kaggle.columns) == list(X.columns), \
    f"Column mismatch!\nExpected: {list(X.columns)}\nGot:      {list(X_test_kaggle.columns)}"
print('\n✓ Column check passed — test features match training features exactly.')


In [ ]:
# ── 5. Predict & save submission ─────────────────────────────────────────────
final_predictions = pipeline.predict(X_test_kaggle)

submission_fe = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': final_predictions
})

print(f'Submission shape: {submission_fe.shape}')  # Must be (418, 2)
print(submission_fe.head(10))

submission_fe.to_csv('submission_feature_engineered.csv', index=False)
print("\n✓ Saved: submission_feature_engineered.csv")
